In [1]:
#01_load_eia923
import os, glob, re, calendar
import pandas as pd

DATA = "."  
months = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"] 
files = glob.glob(os.path.join(DATA, "**", "*Schedules_2_3_4_5*.xlsx"), recursive=True)
files = [f for f in files if not os.path.basename(f).startswith("~$")]
print("find", len(files), "files")
frames = []
for f in sorted(files):
    year = int(re.search(r"f923_(\d{4})", f).group(1))
    df = pd.read_excel(f, sheet_name=0, header=5)
    df.columns = [" ".join(str(c).split()) for c in df.columns]
    wind = df[(df["Reported Fuel Type Code"] == "WND") & (df["Reported Prime Mover"] == "WT")]
    cols = ["Plant Id", "Plant State"] + [f"Netgen {m}" for m in months]
    wind = wind[cols].rename(columns={f"Netgen {m}": i + 1 for i, m in enumerate(months)})
    wind[list(range(1, 13))] = wind[list(range(1, 13))].apply(pd.to_numeric, errors="coerce")
    wind = wind.groupby(["Plant Id", "Plant State"], as_index=False).sum()
    wind.insert(0, "Year", year)
    frames.append(wind)
    print(year, len(wind), "wind plants")
netgen = pd.concat(frames)
netgen.to_csv(os.path.join(DATA, "wind_netgen_monthly.csv"), index=False)
netgen.head()

find 5 files
2021 1278 wind plants
2022 1312 wind plants
2023 1342 wind plants
2024 1354 wind plants
2025 1346 wind plants


,Year,Plant Id,Plant State,1,2,3,4,5,6,7,8,9,10,11,12
0,2021,1,AK,11.609,8.836,11.011,8.745,10.329,10.673,12.572,12.738,13.051,13.182,7.490,13.764
1,2021,90,AK,184.495,140.449,175.033,139.011,164.189,169.653,199.846,202.478,207.460,209.538,119.062,218.786
2,2021,508,CO,1137.468,764.047,1173.504,1138.345,1056.580,934.717,966.945,1026.182,979.891,1075.523,1098.927,1391.871
3,2021,692,WY,2089.526,1788.592,1386.661,1428.685,1204.447,998.867,789.284,1119.544,1097.410,1567.205,2118.211,2505.568
4,2021,944,IL,492.000,511.000,858.000,617.000,572.000,346.000,215.000,218.000,397.000,450.000,772.000,711.000


In [ ]:
#02_merge_eia860
def read_860(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet, header=1)
    df.columns = [" ".join(str(c).split()) for c in df.columns]
    return df
caps = []
for gfile in sorted(glob.glob(os.path.join(DATA, "**", "3_1_Generator*.xlsx"), recursive=True)):
    year = int(re.search(r"Y(\d{4})", os.path.basename(gfile)).group(1))
    pfile = glob.glob(os.path.join(os.path.dirname(gfile), "2___Plant*.xlsx"))[0]
    gen = read_860(gfile, "Operable")
    plant = read_860(pfile, 0)
    gen = gen[gen["Prime Mover"] == "WT"].copy()
    gen["Operating Year"] = pd.to_numeric(gen["Operating Year"], errors="coerce")
    cap = gen.groupby("Plant Code", as_index=False).agg(
        Capacity_MW=("Nameplate Capacity (MW)", "sum"),
        Online_Year=("Operating Year", "min"))
    cap = cap.merge(plant[["Plant Code", "Latitude", "Longitude"]], on="Plant Code", how="left")
    cap["Year"] = year
    caps.append(cap)
    print(year, len(cap), "wind plants in EIA-860")
cap = pd.concat(caps).rename(columns={"Plant Code": "Plant Id"})
long = netgen.melt(id_vars=["Year", "Plant Id", "Plant State"],
                   var_name="Month", value_name="Netgen_MWh")
long["Month"] = long["Month"].astype(int)
df = long.merge(cap, on=["Year", "Plant Id"])
df["Hours"] = [calendar.monthrange(y, m)[1] * 24 for y, m in zip(df["Year"], df["Month"])]
df["CF"] = df["Netgen_MWh"] / (df["Capacity_MW"] * df["Hours"])
df["New_Plant"] = df["Online_Year"] >= df["Year"]
df = df[(df["CF"] >= 0) & (df["CF"] <= 1)]
df.to_csv(os.path.join(DATA, "wind_plants_cf.csv"), index=False)
print(len(df), "lines total")
df.groupby("Month")["CF"].mean().round(3)

2021 1278 wind plants in EIA-860
2022 1299 wind plants in EIA-860
